<a href="https://colab.research.google.com/github/VeXtronics/Team-2026/blob/main/VeXtronicsCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous Farming: Corn Row Alignment Scoring
We built a machine learning system that looks at top-down photos of a corn field, finds every individual corn plant, measures *where* each plant is and *which way it's leaning (growing)*, and then scores how well each plant is aligned with the row it was planted in. The output is two numbers per plant (a positional error in pixels and an angular error in degrees) rolled up into a 0–100 alignment score, with per-row averages.

## The five deliverables, mapped to sections of this notebook
| # | Deliverable | Where in notebook |
|---|-------------|-------------------|
| 1 | Detect individual corn plants in an image | §4 Training, §5 Evaluation |
| 2 | Estimate position and orientation of each plant | §6 Inference |
| 3 | Define the optimal growth line per row | §8 Optimal line |
| 4 | Compute the deviation from that line | §9 Scoring |
| 5 | Produce row-level and plant-level alignment scores | §9 Scoring, §11 Batch export |

## How it works, at a system level
1. **Camera → pixels.** A top-down image of the field.
2. **Detection model → list of plants.** A neural network (YOLO11-pose) looks at the image and returns, for each plant it finds, a bounding box plus three landmark points: the stem center, the tip of the left leaf, and the tip of the right leaf.
3. **Post-processing → measurements.** We take those three points and compute (a) the plant's position = stem location, and (b) the plant's orientation = direction of the line through the two leaf tips.
4. **Row fitting → reference lines.** We group plants by row and either use a known reference line (if we know exactly where the seeds were planted) or we fit a best-fit line through the detected stems.
5. **Scoring → numbers.** For each plant: how far is it from its row's reference line? How rotated is its leaf axis from the expected orientation? We combine these into a single 0–100 score.

---
# 1. Setup

Before doing anything interesting, we need to:

1. Confirm the GPU is working. Training on a CPU would take days.
2. Set the project folder so all outputs land in one place.
3. Import the libraries we'll use throughout.

## Before running this notebook, install these packages (once, from your terminal):

```
pip install ultralytics scikit-learn matplotlib opencv-python
```

If you see a CUDA / GPU-related error later, you may also need the GPU build of PyTorch:

```
pip uninstall torch torchvision -y
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch  # PyTorch — the deep learning framework YOLO is built on

# torch.cuda.is_available() returns True if an NVIDIA GPU + compatible drivers are installed.
# The `assert` line crashes loudly if this returns False, so you find out NOW, not 3 hours into training.
assert torch.cuda.is_available(), "No GPU detected. Check CUDA install / PyTorch version."

# Print GPU model name (useful to confirm you're using the right one if you have multiple).
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
import os
from pathlib import Path
PROJECT_ROOT = Path(r"C:\Users\atpou\projects\T1_Corn_Alignment")

# Create the folder if it doesn't already exist. exist_ok=True means "don't error if it's already there".
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Change the working directory so relative paths in later cells resolve here.
os.chdir(PROJECT_ROOT)

print(f"Working from: {PROJECT_ROOT}")

In [ ]:
import json          # read/write JSON files
import shutil        # copy / delete directories
import random        # Python's built-in random number generator
import yaml          # read/write YAML files (used for YOLO's config format)
from collections import defaultdict  # a dict that auto-creates missing keys as empty lists — handy for grouping
import numpy as np              # arrays and linear algebra — the workhorse of anything numerical in Python
import cv2                      # OpenCV — image loading, drawing, classical computer vision
import matplotlib.pyplot as plt # plotting — how we visualize images inline
from sklearn.cluster import DBSCAN  # clustering algorithm we'll use for row assignment

SEED = 42
random.seed(SEED)         # seed Python's random
np.random.seed(SEED)      # seed NumPy's random
torch.manual_seed(SEED)   # seed PyTorch's random (for weight init, augmentations, etc.)

# Constants describing our annotation scheme: 3 keypoints per plant, in a fixed order.
NUM_KEYPOINTS = 3
KP_STEM  = 0   # index 0 in the keypoint list = stem center
KP_LEFT  = 1   # index 1 = leaf-tip-left
KP_RIGHT = 2   # index 2 = leaf-tip-right

---
# 2. Dataset preparation
The dataset consists of a folder of corn photos, each paired with a text file that has one line per plant: where the bounding box is, and where the three keypoints are. Both images and labels get split into `train/` and `valid/`

## Annotation format
The dataset is exported from Roboflow in "YOLOv8 Pose" format. Each label file looks like:

```
0  0.437 0.389 0.300 0.183  0.479 0.383 2   0.558 0.303 2   0.303 0.316 2
|  |____bbox (x,y,w,h)____| |_kp0 stem_|   |_kp1 left_|   |_kp2 right|
|                                          visibility flag (2 = visible, 0 = not annotated)
class ID (0 = corn)
```

All coordinates are **normalized** to [0, 1], making the labels resolution-independent.

## Expected folder structure

Unzip the Roboflow export into `PROJECT_ROOT/raw_dataset/` so it looks like this:

```
raw_dataset/
|-- train/images/*.jpg   <- training images
|-- train/labels/*.txt   <- one label file per image, same base name
|-- valid/images/*.jpg   <- validation images (model never trains on these)
|-- valid/labels/*.txt
|-- data.yaml            <- we ignore this and generate our own in section 3
```

Now, before proceeding, we need to:
1. Point to the raw dataset and count what we have
2. Canonicalize keypoint order
3. Eyeball a few samples (sanity check)

In [ ]:
# Where the raw Roboflow export lives
RAW_DATASET = PROJECT_ROOT / 'raw_dataset'
assert RAW_DATASET.exists(), f"Unzip your Roboflow export into {RAW_DATASET}"

# Walk the expected subfolders and print counts
for split in ['train', 'valid', 'test']:   # 'test' is optional — many exports skip it
    img_dir = RAW_DATASET / split / 'images'
    lbl_dir = RAW_DATASET / split / 'labels'
    if img_dir.exists():
        # .glob('*') returns every file; len(list(...)) counts them
        n_img = len(list(img_dir.glob('*')))
        n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f"{split:6s}  images={n_img:4d}  labels={n_lbl:4d}")
        # If images and labels don't match, something is wrong with the export
        if n_img != n_lbl:
            print(f"  WARNING: image count and label count don't match for {split}!")

In [ ]:
def canonicalize_label_file(src_path: Path, dst_path: Path):
    """
    Read one YOLO pose label file and rewrite it with:
      - Keypoint 0 = the point closest to the bbox center (the stem)
      - Keypoint 1 = the remaining point with smaller x (left)
      - Keypoint 2 = the remaining point with larger x  (right)
    """
    out_lines = []  # we'll build the new file line by line

    with open(src_path) as f:  # open the source label file for reading
        for line in f:
            # Each line has: class cx cy w h kp0x kp0y kp0v kp1x kp1y kp1v kp2x kp2y kp2v
            parts = line.strip().split()

            # Sanity check: do we have enough numbers? (1 class + 4 bbox + 3*3 keypoints = 14)
            if len(parts) < 5 + NUM_KEYPOINTS * 3:
                out_lines.append(line.rstrip())  # malformed line — keep as-is, don't crash
                continue

            cls = parts[0]  # class ID as string ('0' for corn)
            cx, cy, w, h = map(float, parts[1:5])  # bounding box center (cx,cy) and size (w,h)

            # Parse the three keypoints into (x, y, visibility) tuples
            kps = []
            for i in range(NUM_KEYPOINTS):
                base = 5 + i * 3    # where this keypoint's 3 numbers start in the line
                x = float(parts[base])
                y = float(parts[base + 1])
                v = float(parts[base + 2])
                kps.append((x, y, v))

            # Only consider keypoints that are actually annotated (visibility > 0).
            # Sometimes a leaf is hidden and the annotator marks v=0 to skip it.
            visible = [(i, k) for i, k in enumerate(kps) if k[2] > 0]
            if len(visible) == 0:
                out_lines.append(line.rstrip())  # nothing annotated — leave it
                continue

            # --- Step 1: find the stem = whichever keypoint is closest to the bbox center ---
            def dist_to_centroid(k):
                # squared distance (no need for sqrt since we only compare distances)
                return (k[0] - cx) ** 2 + (k[1] - cy) ** 2

            # min() with a key function picks the item with the smallest value of that function
            stem_idx, stem_kp = min(visible, key=lambda iv: dist_to_centroid(iv[1]))

            # The other two keypoints are the tips
            tip_items = [(i, k) for i, k in enumerate(kps) if i != stem_idx]

            # --- Step 2: sort tips by x-coordinate so left is always index 1 ---
            def tip_sort_key(ik):
                _, k = ik
                # If a tip isn't annotated, push it to the end with a large sentinel value
                if k[2] == 0:
                    return float('inf')
                return k[0] - stem_kp[0]  # x relative to the stem

            tip_items.sort(key=tip_sort_key)
            left_kp  = tip_items[0][1]  # smaller x → left tip
            # Handle the edge case where only one tip was annotated
            right_kp = tip_items[1][1] if len(tip_items) > 1 else (0.0, 0.0, 0.0)

            # --- Reassemble in canonical order: [stem, left, right] ---
            new_kps = [stem_kp, left_kp, right_kp]
            kp_str = ' '.join(f"{x:.6f} {y:.6f} {int(v)}" for (x, y, v) in new_kps)
            out_lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} {kp_str}")

    # Write the cleaned file to its destination
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    with open(dst_path, 'w') as f:
        f.write('\n'.join(out_lines) + '\n')


# We write the cleaned dataset into a separate folder so the raw export is untouched.
# This means you can re-run this cell safely without losing the original labels.
DATASET_ROOT = PROJECT_ROOT / 'dataset'

# If the cleaned dataset already exists from a previous run, delete it first.
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)

# Walk each split (train/valid/test) and process every label file
for split in ['train', 'valid', 'test']:
    src_img = RAW_DATASET / split / 'images'
    src_lbl = RAW_DATASET / split / 'labels'
    if not src_img.exists():  # skip splits that don't exist (e.g. if no test set)
        continue

    dst_img = DATASET_ROOT / split / 'images'
    dst_lbl = DATASET_ROOT / split / 'labels'
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    # Copy images (not symlinks, because Windows symlinks require admin permissions)
    for img in src_img.iterdir():
        tgt = dst_img / img.name
        if not tgt.exists():
            shutil.copy2(img, tgt)  # copy2 preserves metadata like modification time

    # Process and rewrite every label file
    for lbl in src_lbl.glob('*.txt'):
        canonicalize_label_file(lbl, dst_lbl / lbl.name)

    print(f"{split:6s}  done")

In [ ]:
def draw_annotation(img, label_path):
    """
    Takes an image and its label file, and returns a copy of the image
    with the bounding box and keypoints drawn on top.
    Used for visual debugging only.
    """
    H, W = img.shape[:2]  # image height, width in pixels
    out = img.copy()      # don't modify the original

    # BGR colors (OpenCV convention) for each keypoint
    COLORS = {
        KP_STEM:  (255, 0, 0),    # red
        KP_LEFT:  (0, 200, 255),  # cyan-ish
        KP_RIGHT: (255, 200, 0),  # orange-ish
    }
    NAMES = {KP_STEM: 'stem', KP_LEFT: 'L', KP_RIGHT: 'R'}

    if not label_path.exists():
        return out  # no labels — just return the raw image

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5 + NUM_KEYPOINTS * 3:
                continue

            # Convert normalized bbox [0,1] back to absolute pixels for drawing
            cx, cy, w, h = map(float, parts[1:5])
            x1 = int((cx - w/2) * W)
            y1 = int((cy - h/2) * H)
            x2 = int((cx + w/2) * W)
            y2 = int((cy + h/2) * H)
            cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)  # green bbox

            # Draw each keypoint
            for i in range(NUM_KEYPOINTS):
                base = 5 + i * 3
                kx = float(parts[base]) * W      # denormalize x
                ky = float(parts[base + 1]) * H  # denormalize y
                v = float(parts[base + 2])       # visibility flag
                if v > 0:  # only draw if actually annotated
                    cv2.circle(out, (int(kx), int(ky)), 5, COLORS[i], -1)
                    cv2.putText(out, NAMES[i], (int(kx) + 6, int(ky) - 6),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLORS[i], 1)
    return out


# Pick 6 random training images to spot-check
train_imgs = sorted((DATASET_ROOT / 'train' / 'images').iterdir())
sample = random.sample(train_imgs, min(6, len(train_imgs)))

# Set up a 2x3 grid of subplots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img_path in zip(axes.ravel(), sample):
    # OpenCV loads images as BGR; matplotlib expects RGB — convert
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    lbl = DATASET_ROOT / 'train' / 'labels' / (img_path.stem + '.txt')
    ax.imshow(draw_annotation(img, lbl))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.suptitle("Sanity check: bbox + canonicalized keypoints", fontsize=13)
plt.tight_layout()
plt.show()

---
# 3. YOLO Configuration File

## What is `data.yaml`?

YOLO reads training configuration from a small text file in YAML format. We tell it three critical things:

| Setting | What it means |
|---------|---------------|
| `path`, `train`, `val` | Where the images are |
| `kpt_shape: [3, 3]` | Each plant has 3 keypoints, each with 3 values (x, y, visibility) |
| `flip_idx: [0, 2, 1]` | When training flips an image horizontally, remap keypoints: stem stays (0), left becomes right (1→2) and right becomes left (2→1) |
| `names: ['corn']` | We only have one class — "corn" |

## Why `flip_idx` matters

Horizontal flipping is a free way to double the effective size of your dataset — a mirrored corn plant is still a valid corn plant. But if you flip the image without also swapping the L/R keypoint labels, the model sees what's visually a "left leaf" labeled as "right" — broken supervision. `flip_idx` tells YOLO to swap them automatically.

In [ ]:
# Compose the config dictionary in Python first, then dump it to YAML
data_yaml_path = DATASET_ROOT / 'data.yaml'

data_cfg = {
    # Absolute path to the dataset root
    'path': str(DATASET_ROOT),
    # Relative paths to train and val image folders (YOLO finds the matching labels automatically)
    'train': 'train/images',
    'val':   'valid/images',
    # Optional test split — include only if it exists
    'test':  'test/images' if (DATASET_ROOT / 'test' / 'images').exists() else None,

    # Keypoint shape: [num_keypoints, num_values_per_keypoint]
    # 3 keypoints, each with (x, y, visibility) = 3 values
    'kpt_shape': [NUM_KEYPOINTS, 3],

    # When horizontal flip augmentation triggers, this list says:
    # "keypoint index 0 becomes index 0 (stem — unchanged)"
    # "keypoint index 1 becomes index 2 (left tip → right tip)"
    # "keypoint index 2 becomes index 1 (right tip → left tip)"
    'flip_idx':  [KP_STEM, KP_RIGHT, KP_LEFT],

    # Just one class — no weeds, no other crops (for this project)
    'names': ['corn'],
}

# Remove any entries where the value is None (dict comprehension — filters {k:v for ...})
data_cfg = {k: v for k, v in data_cfg.items() if v is not None}

# Write the YAML file
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

# Print so you can see what's in it
print(open(data_yaml_path).read())

---
# 4. YOLO11-pose model Training

## What we're actually training

YOLO is a family of object detection neural networks. YOLO**-pose** variants extend this to also predict keypoints within each detected object — exactly what we need.

We start from a pre-trained model (weights downloaded automatically — trained by Ultralytics on the COCO dataset of 100K+ human pose images) and **fine-tune** it on our 500 corn images. This is transfer learning: the model already knows how to "look" at images and find objects; we're teaching it to specialize in finding corn.

## Why train two sizes

| Model | Params | Speed | Accuracy |
|-------|--------|-------|----------|
| YOLO11n-pose | 3M | Fast | Good |
| YOLO11s-pose | 10M | Slower | Better |

Nano is fast enough for real-time use on embedded hardware. Small is more accurate but ~3× larger. Training both lets us pick the right trade-off once we see how well each performs.

## Key training hyperparameters

| Parameter | Meaning |
|-----------|---------|
| `epochs=200` | Look at the whole dataset 200 times |
| `imgsz=640` | Resize every image to 640×640 before training |
| `batch=16` | Process 16 images at a time (lower if you run out of GPU memory) |
| `patience=40` | Stop early if validation score hasn't improved in 40 epochs (saves time) |
| `pose=30.0` | Weight the keypoint-accuracy loss heavily (default is 12) — keypoints matter more than bbox for us |

## 4.1 Weather-robust augmentation

### What is augmentation?

During training, YOLO randomly modifies each image before feeding it to the model: rotate it, change the colors, add random noise. This teaches the model to be invariant to these variations — it learns what "corn" looks like regardless of lighting, angle, etc.

### Why we use aggressive settings

The project brief calls out rain and sunshine as deployment concerns. We can't collect data in every weather condition, but we can simulate them via augmentation. Strong brightness and HSV jitter simulate sunlight direction and intensity. Random erasing simulates leaves partially occluding each other, rain streaks, or dirt on the camera.

In [ ]:
# Augmentation parameters passed to YOLO's trainer.
# Values are stronger than defaults because our dataset is small and we want robustness.
AUG = dict(
    # Color jitter — simulates different sun angles / times of day
    hsv_h=0.02,        # hue shift (±2%) — small, otherwise plants stop looking green
    hsv_s=0.70,        # saturation (±70%) — strong, simulates overcast vs. direct sun
    hsv_v=0.50,        # brightness (±50%) — strong, simulates shadow vs. bright sun

    # Geometric jitter — simulates camera not being perfectly level/centered
    degrees=15,        # rotation up to ±15°
    translate=0.10,    # translation up to ±10% of image size
    scale=0.40,        # scale up to ±40% — accounts for varying camera height
    shear=3,           # shear up to ±3° — tiny warping
    perspective=0.0005,  # perspective distortion — very small

    # Flipping
    fliplr=0.5,        # 50% chance of horizontal flip (uses flip_idx to swap L/R keypoints)
    flipud=0.0,        # never vertical flip — top-down corn has a consistent orientation

    # Advanced augmentation
    mosaic=1.0,        # always combine 4 training images into one — excellent for small datasets
    mixup=0.10,        # 10% chance to blend two images — regularization
    erasing=0.30,      # 30% chance to erase a random rectangle — simulates occlusion
)
# Dictionary variable that gets unpacked into model.train(...) in the next cells.

## 4.2 Train YOLO11n-pose

This is the fast model (expect 1 hour of training on a decent GPU). Watch the output, you'll see training loss decreasing each epoch, and validation metrics (mAP50, pose mAP50) increasing. If training loss decreases but validation stays flat, the model is memorizing instead of learning - but the augmentation and early stopping should prevent that.

In [ ]:
from ultralytics import YOLO  # YOLO is the main class we use for training and inference

# Load the nano pose model with pre-trained weights.
# If 'yolo11n-pose.pt' isn't already downloaded, Ultralytics fetches it the first time.
model_n = YOLO('yolo11n-pose.pt')

# Kick off training.
# **AUG unpacks our augmentation dict into keyword arguments.
results_n = model_n.train(
    data=str(data_yaml_path),   # path to data.yaml we wrote in section 3
    epochs=200,                 # max training passes through the dataset
    imgsz=640,                  # input resolution
    batch=16,                   # mini-batch size (reduce to 8 or 4 if you see CUDA OOM errors)
    patience=40,                # early-stop after 40 epochs of no improvement
    device=0,                   # GPU index (0 = first GPU)
    project=str(PROJECT_ROOT / 'runs'),  # where to save logs/weights
    name='yolo11n_pose_corn',   # subfolder name for this run
    exist_ok=True,              # overwrite if the run folder already exists
    pose=30.0,                  # keypoint regression loss weight (default 12)
    kobj=2.0,                   # keypoint objectness loss weight
    **AUG,                      # unpack augmentation settings
)
# When this finishes, you'll have weights at: runs/yolo11n_pose_corn/weights/best.pt

## 4.3 Train YOLO11s-pose

Same as above but with a larger model. It is around 2-3× slower to train. Skip this cell if you're short on time - nano is usually good enough.

In [ ]:
# Same approach, bigger model
model_s = YOLO('yolo11s-pose.pt')
results_s = model_s.train(
    data=str(data_yaml_path),
    epochs=200,
    imgsz=640,
    batch=16,                # if you hit out-of-memory errors, try batch=8
    patience=40,
    device=0,
    project=str(PROJECT_ROOT / 'runs'),
    name='yolo11s_pose_corn',
    exist_ok=True,
    pose=30.0,
    kobj=2.0,
    **AUG,
)

---
# 5. Trained Model Evaluation
We look at two kinds of metrics:

## Detection-quality metrics (mAP)

**mAP = mean Average Precision**, it asks: "across all IoU thresholds (how much bbox overlap counts as a correct detection), how many plants did the model find correctly and at what confidence?"

- **mAP50** — lenient, counts a detection as correct if IoU ≥ 50%. Typical "works well" threshold: > 0.85.
- **mAP50-95** — strict, averages over IoU thresholds from 50% to 95%. Typical "works well": > 0.60.

Both exist for boxes and for keypoints. Keypoint mAP is more relevant for us since the downstream alignment math lives on the keypoints.

We will use validation mAP for YOLO to track the best model seen so far and save it as best.pt. That's what will evaluate further with Pixel error.

## Per-keypoint pixel error (direct, interpretable)

For each validation image, we run the model, match each prediction to its ground-truth by bounding-box overlap, and compute the pixel distance between predicted and actual keypoint locations. Averaging over all matched plants gives us interpretable numbers.

### What to look for

- **Stem error** should be the smallest — it's near the plant center, visually unambiguous.
- **Leaf tip errors** tend to be larger — leaves are thin, and exact "tip" can be ambiguous.
- **p95 (95th percentile)** tells you worst-case behavior. If median is 3 pixels but p95 is 40 pixels, you have rare catastrophic failures you should inspect.

As a rule of thumb: stem pixel error should be less than `POS_TOLERANCE_PX / 3` (we set `POS_TOLERANCE_PX = 30` in section 9, so aim for stem error < 10 pixels) to keep scoring noise small compared to real planting noise.

In [ ]:
RUNS_DIR = PROJECT_ROOT / 'runs'
BEST_N = RUNS_DIR / 'yolo11n_pose_corn' / 'weights' / 'best.pt'
BEST_S = RUNS_DIR / 'yolo11s_pose_corn' / 'weights' / 'best.pt'

# Prefer the `s` model if we trained it; otherwise fall back to `n`.
BEST = BEST_S if BEST_S.exists() else BEST_N
print(f"Evaluating: {BEST}")

# Reload the best checkpoint (we could also keep using `model_n`/`model_s` directly;
# this makes the cell self-contained so you can re-run evaluation without re-training).
model = YOLO(str(BEST))

# model.val() runs one pass over the validation set and returns all the standard metrics.
# conf=0.001 means accept all predictions (we compute precision/recall across the whole confidence range).
# iou=0.6 is the threshold for non-max suppression (merging overlapping detections).
metrics = model.val(data=str(data_yaml_path), imgsz=640, conf=0.001, iou=0.6)

print(f"\nBox mAP50:      {metrics.box.map50:.3f}  (higher is better, >0.85 is good)")
print(f"Box mAP50-95:   {metrics.box.map:.3f}    (higher is better, >0.60 is good)")
print(f"Pose mAP50:     {metrics.pose.map50:.3f} (keypoint localization quality)")
print(f"Pose mAP50-95:  {metrics.pose.map:.3f}")

In [ ]:
def iou_xyxy(a, b):
    """
    Compute Intersection-over-Union between two axis-aligned bounding boxes.
    Each box is [x1, y1, x2, y2] (top-left and bottom-right corners).
    Returns a value in [0, 1]. Higher = more overlap = more likely the same object.
    """
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    # Intersection box
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1); ih = max(0, iy2 - iy1)
    inter = iw * ih

    # Union = sum of areas - intersection
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0


# Accumulate per-keypoint pixel errors across the whole validation set
val_imgs = sorted((DATASET_ROOT / 'valid' / 'images').iterdir())
errors = {KP_STEM: [], KP_LEFT: [], KP_RIGHT: []}

for img_path in val_imgs:
    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]

    # Load ground truth for this image
    lbl_path = DATASET_ROOT / 'valid' / 'labels' / (img_path.stem + '.txt')
    if not lbl_path.exists():
        continue

    # Parse GT boxes and keypoints (denormalize to pixels)
    gts = []
    with open(lbl_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 5 + NUM_KEYPOINTS * 3:
                continue
            cx, cy, w, h = map(float, p[1:5])
            box = [(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H]
            kps = []
            for i in range(NUM_KEYPOINTS):
                base = 5 + i * 3
                kps.append((float(p[base]) * W, float(p[base + 1]) * H, float(p[base + 2])))
            gts.append((box, kps))

    # Run the model
    preds = model.predict(str(img_path), conf=0.25, iou=0.5, verbose=False)[0]
    if preds.boxes is None or len(preds.boxes) == 0:
        continue  # no detections — all GTs are "missed"; skip for error calc

    pred_boxes = preds.boxes.xyxy.cpu().numpy()     # (N, 4) array of boxes in pixels
    pred_kps   = preds.keypoints.xy.cpu().numpy()   # (N, 3, 2) array of keypoints in pixels

    # For each GT plant, match to the predicted plant with highest IoU
    for gt_box, gt_kps in gts:
        ious = [iou_xyxy(gt_box, pb) for pb in pred_boxes]
        if not ious or max(ious) < 0.3:
            continue  # no prediction overlapped this GT enough — treat as missed
        best = int(np.argmax(ious))  # index of the best-matching prediction

        # For each keypoint, accumulate the pixel distance between GT and prediction
        for i in range(NUM_KEYPOINTS):
            gx, gy, gv = gt_kps[i]
            if gv == 0:
                continue  # GT didn't annotate this keypoint — skip
            px, py = pred_kps[best, i]
            errors[i].append(np.hypot(px - gx, py - gy))  # Euclidean distance

# Print summary
print(f"{'Keypoint':<14}{'mean px':>10}{'median px':>12}{'p95 px':>10}{'n':>8}")
for i, name in zip([KP_STEM, KP_LEFT, KP_RIGHT], ['stem', 'leaf-left', 'leaf-right']):
    e = np.array(errors[i])
    if len(e):
        print(f"{name:<14}{e.mean():>10.2f}{np.median(e):>12.2f}"
              f"{np.percentile(e, 95):>10.2f}{len(e):>8}")

In [ ]:
# --Sanity Check--
# Look at 4 random validation images with the model's predictions drawn on.

sample_val = random.sample(val_imgs, min(4, len(val_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, img_path in zip(axes.ravel(), sample_val):
    # model.predict returns a Results object; [0] = results for image 0 (we pass one image)
    result = model.predict(str(img_path), conf=0.25, verbose=False)[0]
    # .plot() returns a numpy array with boxes + keypoints already drawn
    annotated = result.plot(kpt_radius=6, kpt_line=True)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')
plt.suptitle("Model predictions on validation images", fontsize=13)
plt.tight_layout()
plt.show()

---
# 6. Inference Wrapper

The raw output of `model.predict()` is a complex object with tensors, confidence scores, etc. For the downstream code (row assignment, scoring), we want a simple list of dictionaries - one per plant - with named fields.

This cell defines a single function `detect_plants(image)` that returns a list like:

```python
[
    {
        'bbox':           [x1, y1, x2, y2],       # in pixels
        'confidence':     0.94,                    # how sure the model is (0 to 1)
        'stem':           (450.2, 312.1),          # (x, y) or None if not confident
        'left_tip':       (420.0, 295.3),
        'right_tip':      (485.7, 301.8),
        'leaf_axis_deg':  12.5,                    # angle of the left-right axis
    },
    ...
]
```

This is the clean interface every later cell uses.

In [ ]:
# Confidence threshold for keypoints: below this, we treat the keypoint as "missing"
# rather than trusting a low-confidence guess. Tune based on section 5.2 results.
KP_CONF_THRESHOLD = 0.3


def axis_angle_deg(p_left, p_right):
    """
    Compute the angle of the axis through the two leaf tips.

    Returns angle in degrees in the range (-90, 90]:
      0 deg   = horizontal (tips are left-right)
      90 deg  = vertical   (one tip above the other)
      -45 deg = diagonal down-right

    Why (-90, 90] instead of (-180, 180]? Because an axis (a line) has no direction.
    An axis at +170 deg and an axis at -10 deg describe the same line.
    We normalize so downstream code doesn't have to worry about equivalent angles.
    """
    if p_left is None or p_right is None:
        return None  # can't compute an axis if either tip is missing

    dx = p_right[0] - p_left[0]
    dy = p_right[1] - p_left[1]
    # arctan2 gives signed angle in (-180, 180]; np.degrees converts radians to degrees
    angle = np.degrees(np.arctan2(dy, dx))

    # Wrap into (-90, 90] — flip by 180 deg if outside that range (same line, opposite direction)
    while angle >   90: angle -= 180
    while angle <= -90: angle += 180
    return angle


def detect_plants(image_or_path, conf=0.25, iou=0.5):
    """
    Run the trained model on one image and return a list of plant dictionaries.

    Parameters
    ----------
    image_or_path : str or np.ndarray
        Path to an image file, or an image array already loaded with cv2.
    conf : float
        Minimum confidence to keep a detection (0 to 1). 0.25 = keep everything that's
        25%+ likely to be a plant. Lower catches more, but with more false positives.
    iou : float
        IoU threshold for non-max suppression (merging overlapping detections).
    """
    # Run inference; [0] because predict returns a list (one per input image)
    r = model.predict(image_or_path, conf=conf, iou=iou, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []  # no plants detected

    # Pull tensors off the GPU and convert to numpy arrays for convenience
    boxes = r.boxes.xyxy.cpu().numpy()   # (N, 4) — each row is [x1, y1, x2, y2]
    confs = r.boxes.conf.cpu().numpy()   # (N,)   — confidence per detection
    kp_xy = r.keypoints.xy.cpu().numpy()  # (N, 3, 2) — (x, y) for each of 3 keypoints
    # Per-keypoint confidence isn't always exposed; fall back to 1.0 if unavailable
    try:
        kp_conf = r.keypoints.conf.cpu().numpy()  # (N, 3)
    except Exception:
        kp_conf = np.ones((len(boxes), NUM_KEYPOINTS))

    # Build our simple list of dictionaries
    plants = []
    for i in range(len(boxes)):

        def kp_or_none(idx):
            """Return keypoint (x,y) tuple or None if confidence is too low."""
            if kp_conf[i, idx] < KP_CONF_THRESHOLD:
                return None
            return (float(kp_xy[i, idx, 0]), float(kp_xy[i, idx, 1]))

        stem  = kp_or_none(KP_STEM)
        left  = kp_or_none(KP_LEFT)
        right = kp_or_none(KP_RIGHT)

        plants.append({
            'bbox':          boxes[i].tolist(),
            'confidence':    float(confs[i]),
            'stem':          stem,
            'left_tip':      left,
            'right_tip':     right,
            'leaf_axis_deg': axis_angle_deg(left, right),
        })
    return plants


# --- Quick demo ---
demo_path = random.choice(val_imgs)
plants = detect_plants(str(demo_path))
print(f"Detected {len(plants)} plants in {demo_path.name}")
# Print the first 3 plants to confirm the structure
for p in plants[:3]:
    print(f"  conf={p['confidence']:.2f}  stem={p['stem']}  axis={p['leaf_axis_deg']}")

---
# 7. Plant to Row Assignment
## The problem

We have a list of detected plants, each with an (x, y) stem position in the image. We know the field has several parallel rows. We need to say: "this plant belongs to row 0, this one to row 1, ..." so we can score each row separately.

## The approach — 1D clustering on cross-row projections

Imagine looking at the field from the side, along the direction of the rows. The plants collapse onto a 1D line where each row is a tight cluster. That's exactly what we do mathematically: take each stem's 2D position, project it onto the axis perpendicular to the row direction, and now we have 1D coordinates. Plants in the same row land on the same value; plants in different rows land on different values.

We then cluster the 1D positions with DBSCAN — a density-based clustering algorithm. Unlike k-means, DBSCAN doesn't need you to say how many clusters there are in advance; it finds them automatically. It also handles noise (plants that don't belong to any row).

## The row direction

Needs to be known or estimated. In our greenhouse setup, rows run front-to-back in the image, so we know it roughly. The helper function `estimate_row_direction_deg` estimates it from the data via PCA (Principal Component Analysis): the rows collectively lie along the direction of maximum variance, so the first principal component of the stem positions is the row direction.

In [ ]:
def estimate_row_direction_deg(plants):
    """
    Estimate the row direction from the detected stem positions using PCA.

    Rows are long and narrow — the direction along the rows has the most variance.
    So the first principal component of the stem (x, y) cloud is the row direction.

    Returns the angle in degrees. 0 deg = rows are horizontal; 90 deg = rows are vertical.
    """
    # Collect stems into an (N, 2) array
    stems = np.array([p['stem'] for p in plants if p['stem'] is not None])
    if len(stems) < 3:
        return 0.0  # not enough points to estimate; default to horizontal

    # Subtract the centroid (center the data) — PCA requires this
    centered = stems - stems.mean(axis=0)

    # Singular Value Decomposition gives us the principal components.
    # vh[0] is the first principal component — the direction of max variance.
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    vx, vy = vh[0]

    # Convert vector (vx, vy) to an angle
    return float(np.degrees(np.arctan2(vy, vx)))


def assign_rows(plants, row_direction_deg, eps_px=40, min_samples=1):
    """
    Group detected plants into rows.

    Parameters
    ----------
    plants : list of dicts
        Output from detect_plants().
    row_direction_deg : float
        Known or estimated angle of the rows, in degrees.
    eps_px : float
        DBSCAN parameter: two plants within eps_px (in the cross-row direction)
        are considered part of the same row. Set to roughly half the inter-row spacing.
    min_samples : int
        DBSCAN parameter: minimum number of plants for a cluster to count as a row.
        Set to 2 so single-plant "rows" are flagged as noise (row_id = -1).

    Modifies `plants` in-place — adds a 'row_id' field to each plant.
    Returns the modified list.
    """
    # Unit vector perpendicular to the row direction (the "cross-row axis")
    theta = np.radians(row_direction_deg + 90)  # +90 deg rotates the direction by a right angle
    perp = np.array([np.cos(theta), np.sin(theta)])

    # Project each stem onto the perpendicular axis; this gives a 1D coordinate per plant
    projections, valid_idx = [], []
    for i, p in enumerate(plants):
        if p['stem'] is None:
            continue
        projections.append(np.dot(p['stem'], perp))  # dot product = projection
        valid_idx.append(i)

    # sklearn's DBSCAN expects (N, 1) input for 1D clustering
    projections = np.array(projections).reshape(-1, 1)
    if len(projections) == 0:
        return plants

    # Run DBSCAN. Returns a label per point:
    #   -1 = noise (doesn't belong to any cluster)
    #    0, 1, 2, ... = cluster IDs
    labels = DBSCAN(eps=eps_px, min_samples=min_samples).fit_predict(projections)

    # DBSCAN assigns cluster IDs in the order it encounters them, which can be random.
    # Relabel so row 0 is the topmost row, row 1 is next, etc. (ordered by projection value).
    unique_labels = sorted(set(labels) - {-1})  # -1 is noise, exclude it
    mean_proj = {lb: projections[labels == lb].mean() for lb in unique_labels}
    order = sorted(unique_labels, key=lambda lb: mean_proj[lb])
    remap = {lb: new for new, lb in enumerate(order)}
    remap[-1] = -1  # noise stays as noise

    # Write the row_id back onto each plant dict
    for idx_in_valid, global_idx in enumerate(valid_idx):
        plants[global_idx]['row_id'] = remap[int(labels[idx_in_valid])]

    # Plants with no stem (and therefore skipped above) get row_id = -1
    for p in plants:
        p.setdefault('row_id', -1)

    return plants


# --- Demo on our previous detection ---
row_dir_deg = estimate_row_direction_deg(plants)
plants = assign_rows(plants, row_direction_deg=row_dir_deg)
n_rows = len(set(p['row_id'] for p in plants if p['row_id'] >= 0))
print(f"Estimated row direction: {row_dir_deg:.1f} deg")
print(f"Assigned {sum(1 for p in plants if p['row_id']>=0)} plants into {n_rows} rows")

---
# 8. Optimal Line per Row Definition

## The key insight

In our greenhouse trial, the seeds were deliberately hand-planted along straight perpendicular lines. So the **optimal line is the row they were supposed to land on** - we know it by construction. We're not trying to discover it from the data; we're measuring how far the plants drifted from it.

## Two ways to supply the optimal line

1. **If you know exactly where the rows were planted** (e.g., from the physical layout or a marking line in the image): use `OptimalLine.from_point_angle()` and hard-code the values. This is the most accurate.

2. **If you don't**: fit a line through the detected stems in each row using **RANSAC**. RANSAC stands for Random Sample Consensus. Unlike ordinary least-squares fitting, it ignores outliers — the few plants that were planted badly off-line. Why? Because those outliers are exactly what we want to *score*, not fit our reference to.

## Why RANSAC, not least-squares

Imagine a row that was planted mostly straight but one seed drifted 10 cm off. Least-squares fitting would tilt the line toward that outlier, giving every other plant a small fake deviation to compensate. RANSAC picks a line that fits the majority of points well and marks the outlier as an outlier. The outlier gets its deserved high deviation score in the end.

In [ ]:
from dataclasses import dataclass
from sklearn.linear_model import RANSACRegressor


@dataclass
class OptimalLine:
    """
    Represents a line in image coordinates as a point + unit direction vector.

    Why store it this way instead of slope + intercept? Because slope-intercept
    breaks down for vertical lines (infinite slope). Point + direction works for
    any line orientation.

    Attributes
    ----------
    px, py : float
        A point on the line, in image pixels.
    dx, dy : float
        The direction vector of the line (unit length: dx*dx + dy*dy = 1).
    """
    px: float
    py: float
    dx: float
    dy: float

    @classmethod
    def from_point_angle(cls, px, py, angle_deg):
        """Construct a line from a point and an angle in degrees."""
        t = np.radians(angle_deg)
        return cls(px, py, np.cos(t), np.sin(t))

    @classmethod
    def from_ransac(cls, stems_xy, residual_threshold=15.0, random_state=SEED):
        """
        Fit a line to a cloud of stem (x, y) points using RANSAC.

        residual_threshold (in pixels) = how far from the line a point can be
        while still counting as an inlier. Set this to your planting tolerance.
        """
        stems = np.asarray(stems_xy)
        if len(stems) < 2:
            # Degenerate — only one point; return a horizontal line through it
            return cls(stems[0, 0], stems[0, 1], 1.0, 0.0)

        X, y = stems[:, 0:1], stems[:, 1]

        # Detect near-vertical rows and handle them separately.
        # A near-vertical row has very little x-spread — fitting y(x) is unstable.
        # We fit x(y) instead and then flip the result back.
        if X.std() < y.std() * 0.2:
            # Near-vertical case: regress x on y
            ry = RANSACRegressor(residual_threshold=residual_threshold,
                                 random_state=random_state).fit(stems[:, 1:2], stems[:, 0])
            m = ry.estimator_.coef_[0]     # slope of x = m*y + c
            c = ry.estimator_.intercept_   # intercept
            n = np.hypot(m, 1)              # magnitude for normalization
            # Direction vector (m, 1) normalized; passes through (c, 0)
            return cls(c, 0.0, m / n, 1.0 / n)
        else:
            # Normal case: regress y on x
            rx = RANSACRegressor(residual_threshold=residual_threshold,
                                 random_state=random_state).fit(X, y)
            m = rx.estimator_.coef_[0]      # slope of y = m*x + c
            c = rx.estimator_.intercept_
            n = np.hypot(1, m)
            # Direction vector (1, m) normalized; passes through (0, c)
            return cls(0.0, c, 1.0 / n, m / n)

    def perpendicular_distance(self, pt):
        """
        Signed perpendicular distance from a point to the line, in pixels.

        Uses the 2D cross-product formula:
          distance = (pt - P) x direction
        where x is the scalar cross product (positive = left of line, negative = right).

        We usually take the absolute value for scoring.
        """
        vx, vy = pt[0] - self.px, pt[1] - self.py
        return vx * self.dy - vy * self.dx

    def angle_deg(self):
        """Return the angle of the line in degrees, normalized to (-90, 90]."""
        a = np.degrees(np.arctan2(self.dy, self.dx))
        while a >   90: a -= 180
        while a <= -90: a += 180
        return a


def fit_optimal_lines(plants, known_lines=None):
    """
    Build a dict mapping row_id -> OptimalLine.

    Parameters
    ----------
    plants : list of plant dicts (with row_id set by assign_rows)
    known_lines : optional dict {row_id: (px, py, angle_deg)}
        If provided, hard-code the line for that row instead of fitting.
        Use this when you know the exact planted layout.
    """
    lines = {}

    # Group stems by row_id
    rows = defaultdict(list)
    for p in plants:
        if p['row_id'] >= 0 and p['stem'] is not None:
            rows[p['row_id']].append(p['stem'])

    for rid, stems in rows.items():
        if known_lines and rid in known_lines:
            # Hard-coded line from layout
            px, py, a = known_lines[rid]
            lines[rid] = OptimalLine.from_point_angle(px, py, a)
        else:
            # Fit with RANSAC from the detected stems
            lines[rid] = OptimalLine.from_ransac(stems)
    return lines


# --- Demo ---
optimal_lines = fit_optimal_lines(plants)
for rid, line in sorted(optimal_lines.items()):
    print(f"Row {rid}: angle={line.angle_deg():+.2f} deg  "
          f"passes through ({line.px:.0f}, {line.py:.0f})")

---
# 9. Alignment Scores and Deviations Computation

## Per-plant deviations

For every plant, we compute two numbers:

- **Positional deviation** — perpendicular pixel distance from the stem to its row's optimal line. Directly measures "how far off is this plant from where it was supposed to be?"
- **Angular deviation** — absolute angle between the plant's leaf axis and the expected orientation. For our perpendicular trial, leaves should grow **across** the row, so the expected leaf axis is 90° off from the row direction. `EXPECTED_LEAF_OFFSET_DEG = 90°`.

## Per-plant score (0 to 100, higher = better)

We turn each deviation into a score in [0, 100]:

```
pos_score(d) = 100 * exp(-d / POS_TOLERANCE_PX)         # exponential decay
ang_score(delta) = 100 * max(0, 1 - delta / ANG_TOLERANCE_DEG)  # linear decay, clamped at 0
plant_score = 0.5 * pos_score + 0.5 * ang_score         # equal weighting
```

Why exponential decay for position? A plant 30 pixels off is bad, but a plant 60 pixels off isn't "twice as bad" — it's "qualitatively wrong." Exponential decay matches that intuition.

Why linear for angle? Because angle has a natural maximum (90° = completely perpendicular = 0 score makes sense); pegging the score at 0 past the tolerance is the simplest defensible rule.

## Per-row score

Average the plant scores within each row. Also report the fraction of plants within tolerance — this is the number a farmer would actually care about: "85% of my plants in row 3 are within spec."

## Tuning knobs

The two constants below control how "strict" the scoring is. **Set these based on what a farmer considers acceptable planting error**, not what the model can achieve. The brief calls this out as an open question — we can only answer it once we see real planting data.

In [ ]:
# --- Scoring thresholds — tune these based on real-world planting tolerance ---
POS_TOLERANCE_PX  = 30.0      # pixel offset at which positional score drops to ~37/100 (1/e)
ANG_TOLERANCE_DEG = 20.0      # angle offset at which angular score hits 0/100
EXPECTED_LEAF_OFFSET_DEG = 90.0  # leaves grow across (perpendicular to) the row


def angular_diff_deg(a, b):
    """
    Smallest unsigned difference between two axis angles, normalized to [0, 90].

    Because an axis has no direction (line, not arrow), 170 deg and -10 deg describe
    the same axis, and their difference should be 0.
    """
    if a is None or b is None:
        return None
    d = abs(a - b) % 180
    return min(d, 180 - d)


def score_plants(plants, optimal_lines,
                 pos_tol=POS_TOLERANCE_PX,
                 ang_tol=ANG_TOLERANCE_DEG,
                 expected_offset=EXPECTED_LEAF_OFFSET_DEG):
    """
    Add scoring fields to each plant dict:
      pos_dev_px, ang_dev_deg, pos_score, ang_score, plant_score
    """
    for p in plants:
        rid = p['row_id']
        # Can't score plants with no row assignment or no stem
        if rid < 0 or rid not in optimal_lines or p['stem'] is None:
            p.update(dict(pos_dev_px=None, ang_dev_deg=None,
                          pos_score=None, ang_score=None, plant_score=None))
            continue

        line = optimal_lines[rid]

        # --- Positional deviation ---
        pos_dev = abs(line.perpendicular_distance(p['stem']))  # perpendicular distance, pixels
        p['pos_dev_px'] = pos_dev
        # Exponential decay: 100 at d=0, ~37 at d=pos_tol, ~14 at d=2*pos_tol
        p['pos_score']  = 100.0 * np.exp(-pos_dev / pos_tol)

        # --- Angular deviation ---
        if p['leaf_axis_deg'] is not None:
            # Expected leaf axis = row direction + 90 deg (perpendicular)
            expected_axis_deg = line.angle_deg() + expected_offset
            # Normalize expected axis to (-90, 90]
            while expected_axis_deg >   90: expected_axis_deg -= 180
            while expected_axis_deg <= -90: expected_axis_deg += 180

            ang_dev = angular_diff_deg(p['leaf_axis_deg'], expected_axis_deg)
            p['ang_dev_deg'] = ang_dev
            # Linear decay: 100 at dev=0, 0 at dev=ang_tol, still 0 beyond
            p['ang_score']   = 100.0 * max(0.0, 1.0 - ang_dev / ang_tol)
        else:
            p['ang_dev_deg'] = None
            p['ang_score']   = None

        # --- Combined score ---
        if p['ang_score'] is not None:
            p['plant_score'] = 0.5 * p['pos_score'] + 0.5 * p['ang_score']
        else:
            # If we couldn't compute angular (e.g., one leaf tip missing),
            # fall back to positional only
            p['plant_score'] = p['pos_score']
    return plants


def row_scores(plants):
    """
    Roll plant-level scores up to per-row summaries.

    For each row, returns:
      n              — number of plants
      mean_score     — average plant_score across the row
      mean_pos_px    — average positional deviation
      max_pos_px     — worst positional deviation (useful for spotting bad plants)
      mean_ang_deg   — average angular deviation
      within_tol_pct — fraction of plants within POS_TOLERANCE_PX
    """
    # Group plants by row
    rows = defaultdict(list)
    for p in plants:
        if p['row_id'] >= 0 and p['plant_score'] is not None:
            rows[p['row_id']].append(p)

    summary = {}
    for rid, plist in rows.items():
        pos_devs = [p['pos_dev_px']  for p in plist if p['pos_dev_px']  is not None]
        ang_devs = [p['ang_dev_deg'] for p in plist if p['ang_dev_deg'] is not None]
        scores   = [p['plant_score'] for p in plist]
        within   = sum(1 for d in pos_devs if d <= POS_TOLERANCE_PX) / max(1, len(pos_devs))

        summary[rid] = dict(
            n=len(plist),
            mean_score=float(np.mean(scores)),
            mean_pos_px=float(np.mean(pos_devs)) if pos_devs else None,
            max_pos_px=float(np.max(pos_devs)) if pos_devs else None,
            mean_ang_deg=float(np.mean(ang_devs)) if ang_devs else None,
            within_tol_pct=100.0 * within,
        )
    return summary


# --- Demo ---
plants = score_plants(plants, optimal_lines)
row_summary = row_scores(plants)

# Nice printout
print(f"{'Row':>4}  {'n':>3}  {'score':>6}  {'mean-pos':>9}  {'max-pos':>8}  "
      f"{'mean-ang':>9}  {'within-tol':>10}")
for rid, s in sorted(row_summary.items()):
    print(f"{rid:>4}  {s['n']:>3}  {s['mean_score']:>6.1f}  "
          f"{(s['mean_pos_px'] or 0):>9.1f}  {(s['max_pos_px'] or 0):>8.1f}  "
          f"{(s['mean_ang_deg'] or 0):>9.1f}  {s['within_tol_pct']:>9.1f}%")

---
# 10. Batch Processing and Results Visualization

We color-code every plant by its alignment score: green = perfect, red = badly off. Draw the optimal line per row in white. This is the picture we show the supervisor to communicate results at a glance.

Afterwards we loop over every image in the test (or validation) set, run the full pipeline, and writes two CSV files:

- **plants.csv** — one row per detected plant, with its deviation and score
- **rows.csv** — one row per detected row, with aggregate stats

Open these in Excel or pandas for any further analysis — box plots of deviation, histograms of score per row, etc.

In [ ]:
def visualize_alignment(img_bgr, plants, optimal_lines):
    """
    Draw bounding boxes, keypoints, and optimal lines on the image.
    Color-code by plant score: green = good, yellow = medium, red = poor.
    """
    H, W = img_bgr.shape[:2]
    out = img_bgr.copy()

    # --- Draw optimal lines first (so plants are drawn on top) ---
    for rid, line in optimal_lines.items():
        # Extend the line across the image by walking a large distance in both directions
        t_vals = np.linspace(-max(W, H), max(W, H), 2)
        xs = line.px + t_vals * line.dx
        ys = line.py + t_vals * line.dy
        cv2.line(out, (int(xs[0]), int(ys[0])), (int(xs[1]), int(ys[1])),
                 (255, 255, 255), 2, cv2.LINE_AA)

    # --- Draw each plant with a score-dependent color ---
    for p in plants:
        if p['stem'] is None:
            continue

        score = p['plant_score'] if p['plant_score'] is not None else 0
        # Map score [0, 100] to a red-green gradient
        # (Note: OpenCV uses BGR order, not RGB)
        g = int(np.clip(score * 2.55,         0, 255))
        r = int(np.clip((100 - score) * 2.55, 0, 255))
        color = (0, g, r)  # BGR: blue=0, green=g, red=r

        # Bounding box
        x1, y1, x2, y2 = map(int, p['bbox'])
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)

        # Stem
        sx, sy = map(int, p['stem'])
        cv2.circle(out, (sx, sy), 5, color, -1)

        # Leaf axis — line segment through the two tips
        if p['left_tip'] and p['right_tip']:
            lx, ly = map(int, p['left_tip'])
            rx, ry = map(int, p['right_tip'])
            cv2.line(out, (lx, ly), (rx, ry), color, 2)
            cv2.circle(out, (lx, ly), 4, (0, 200, 255), -1)  # cyan for left tip
            cv2.circle(out, (rx, ry), 4, (255, 200, 0), -1)  # orange for right tip

        # Score text above the bounding box
        if p['plant_score'] is not None:
            cv2.putText(out, f"{p['plant_score']:.0f}", (x1, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    return out


# --- Run the entire pipeline on one random image ---
demo_path   = random.choice(val_imgs)
img         = cv2.imread(str(demo_path))
plants      = detect_plants(str(demo_path))                               # section 6
row_dir_deg = estimate_row_direction_deg(plants)                          # section 7
plants      = assign_rows(plants, row_direction_deg=row_dir_deg)          # section 7
optimal_lines = fit_optimal_lines(plants)                                 # section 8
plants      = score_plants(plants, optimal_lines)                         # section 9
vis         = visualize_alignment(img, plants, optimal_lines)

plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"{demo_path.name} — {len(plants)} plants, {len(optimal_lines)} rows\n"
          f"White line = optimal row. Plant color = alignment score (green=good, red=poor).")
plt.axis('off')
plt.show()

In [ ]:
import csv  # standard library CSV writer

# Output folder
OUT_DIR = PROJECT_ROOT / 'results'
OUT_DIR.mkdir(exist_ok=True)

plant_csv = OUT_DIR / 'plants.csv'
row_csv   = OUT_DIR / 'rows.csv'

# Column headers for the plants CSV
plant_header = ['image', 'row_id', 'x1', 'y1', 'x2', 'y2', 'conf',
                'stem_x', 'stem_y', 'left_x', 'left_y', 'right_x', 'right_y',
                'leaf_axis_deg', 'pos_dev_px', 'ang_dev_deg',
                'pos_score', 'ang_score', 'plant_score']

# Column headers for the rows CSV
row_header = ['image', 'row_id', 'n', 'mean_score',
              'mean_pos_px', 'max_pos_px', 'mean_ang_deg', 'within_tol_pct']

# Prefer the test set if it exists; otherwise use the validation set
test_dir = DATASET_ROOT / 'test' / 'images'
target_dir = test_dir if test_dir.exists() else DATASET_ROOT / 'valid' / 'images'
target_imgs = sorted(target_dir.iterdir())
print(f"Processing {len(target_imgs)} images from {target_dir}")

# Open both CSVs and loop through every image
with open(plant_csv, 'w', newline='') as fp, open(row_csv, 'w', newline='') as fr:
    pw = csv.writer(fp); pw.writerow(plant_header)
    rw = csv.writer(fr); rw.writerow(row_header)

    for img_path in target_imgs:
        # Full pipeline for this image
        plants = detect_plants(str(img_path))
        if not plants:
            continue
        plants = assign_rows(plants, row_direction_deg=estimate_row_direction_deg(plants))
        lines = fit_optimal_lines(plants)
        plants = score_plants(plants, lines)

        # Write one row per plant
        for p in plants:
            pw.writerow([
                img_path.name, p['row_id'],
                *[f"{v:.1f}" for v in p['bbox']], f"{p['confidence']:.3f}",
                # (x, y) or (NaN, NaN) if missing — the `or` idiom handles None
                *(f"{v:.1f}" for v in (p['stem']      or (np.nan, np.nan))),
                *(f"{v:.1f}" for v in (p['left_tip']  or (np.nan, np.nan))),
                *(f"{v:.1f}" for v in (p['right_tip'] or (np.nan, np.nan))),
                f"{p['leaf_axis_deg']:.1f}" if p['leaf_axis_deg'] is not None else '',
                f"{p['pos_dev_px']:.1f}"   if p['pos_dev_px']   is not None else '',
                f"{p['ang_dev_deg']:.1f}"  if p['ang_dev_deg']  is not None else '',
                f"{p['pos_score']:.1f}"    if p['pos_score']    is not None else '',
                f"{p['ang_score']:.1f}"    if p['ang_score']    is not None else '',
                f"{p['plant_score']:.1f}"  if p['plant_score']  is not None else '',
            ])

        # Write one row per detected row
        for rid, s in row_scores(plants).items():
            rw.writerow([
                img_path.name, rid, s['n'], f"{s['mean_score']:.2f}",
                f"{s['mean_pos_px']:.2f}"  if s['mean_pos_px']  is not None else '',
                f"{s['max_pos_px']:.2f}"   if s['max_pos_px']   is not None else '',
                f"{s['mean_ang_deg']:.2f}" if s['mean_ang_deg'] is not None else '',
                f"{s['within_tol_pct']:.2f}",
            ])

print(f"Wrote {plant_csv}")
print(f"Wrote {row_csv}")

**Live Camera Angle Detection**


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6b — Live Camera Angle Detection
# ─────────────────────────────────────────────────────────────────────────────
# Prerequisites: Section 6 must already have run so `model` and
#               `detect_plants()` / `axis_angle_deg()` are in scope.
#
# Controls while the window is open:
#   Q / Esc  → quit
#   S        → save the current annotated frame as a PNG
# ─────────────────────────────────────────────────────────────────────────────

import cv2
import numpy as np
from pathlib import Path
from datetime import datetime

# ── Config ────────────────────────────────────────────────────────────────────
CAMERA_INDEX   = 0          # 0 = default webcam, 1/2/... for external cameras
CONF_THRESH    = 0.25       # detection confidence cutoff
IOU_THRESH     = 0.45       # NMS threshold
WINDOW_NAME    = "VeXtronics — Live Angle Detection"
SAVE_DIR       = PROJECT_ROOT / "live_snapshots"
SAVE_DIR.mkdir(exist_ok=True)

# ── Drawing helpers ───────────────────────────────────────────────────────────
def draw_angle_arc(frame, stem, angle_deg, radius=30, color=(0, 255, 200)):
    """
    Draw a short arc around the stem to visualise the leaf axis angle.
    The arc spans ±30° of the computed axis so it's easy to read at a glance.
    """
    if stem is None:
        return
    cx, cy = int(stem[0]), int(stem[1])
    # OpenCV ellipse angles are measured clockwise from 3 o'clock
    start_a = -(angle_deg + 30)
    end_a   = -(angle_deg - 30)
    cv2.ellipse(frame, (cx, cy), (radius, radius), 0,
                start_a, end_a, color, 2, cv2.LINE_AA)


def draw_axis_line(frame, left_tip, right_tip, color=(0, 255, 200)):
    """Draw the leaf-axis line between the two keypoint tips."""
    if left_tip is None or right_tip is None:
        return
    cv2.line(frame,
             (int(left_tip[0]),  int(left_tip[1])),
             (int(right_tip[0]), int(right_tip[1])),
             color, 2, cv2.LINE_AA)


def annotate_frame(frame, plants):
    """
    Draw detections on `frame` in-place and return it.

    For every detected plant:
      • Green bounding box
      • Red dot at the stem
      • Cyan dots at the leaf tips
      • Cyan axis line between the tips
      • Angle arc around the stem
      • Text label with the angle value
    """
    for p in plants:
        x1, y1, x2, y2 = map(int, p['bbox'])
        angle = p['leaf_axis_deg']
        stem  = p['stem']

        # ── Bounding box ──────────────────────────────────────────────────
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 220, 0), 2)

        # ── Leaf axis line ────────────────────────────────────────────────
        draw_axis_line(frame, p['left_tip'], p['right_tip'], (0, 220, 220))

        # ── Keypoints ─────────────────────────────────────────────────────
        if stem:
            cv2.circle(frame, (int(stem[0]), int(stem[1])), 5, (0, 0, 220), -1)
        for tip in [p['left_tip'], p['right_tip']]:
            if tip:
                cv2.circle(frame, (int(tip[0]), int(tip[1])), 4, (0, 220, 220), -1)

        # ── Angle arc + text ──────────────────────────────────────────────
        if angle is not None and stem:
            draw_angle_arc(frame, stem, angle)
            label = f"{angle:+.1f} deg"
            cv2.putText(frame, label,
                        (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 220), 2, cv2.LINE_AA)

    return frame


def draw_hud(frame, n_plants, fps):
    """Overlay a heads-up display with plant count and FPS in the top-left corner."""
    h, w = frame.shape[:2]
    # Semi-transparent dark bar
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (260, 50), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)

    cv2.putText(frame, f"Plants: {n_plants}   FPS: {fps:.1f}",
                (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, "Q/Esc=quit  S=save",
                (10, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1, cv2.LINE_AA)


# ── Main camera loop ──────────────────────────────────────────────────────────
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise RuntimeError(
        f"Cannot open camera index {CAMERA_INDEX}. "
        "Try changing CAMERA_INDEX or check that no other app holds the camera."
    )

# Optional: set a larger resolution if your camera supports it
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print(f"Camera opened. Resolution: "
      f"{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}×"
      f"{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
print("Press Q or Esc to quit, S to save a snapshot.")

import time
prev_time = time.time()

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Frame grab failed — is the camera still connected?")
            break

        # ── Inference ─────────────────────────────────────────────────────
        plants = detect_plants(frame,          # pass the numpy array directly
                               conf=CONF_THRESH,
                               iou=IOU_THRESH)

        # ── Draw ──────────────────────────────────────────────────────────
        annotate_frame(frame, plants)

        # FPS counter
        now  = time.time()
        fps  = 1.0 / max(now - prev_time, 1e-6)
        prev_time = now
        draw_hud(frame, len(plants), fps)

        # ── Display ───────────────────────────────────────────────────────
        cv2.imshow(WINDOW_NAME, frame)

        # ── Key handling ──────────────────────────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key in (ord('q'), ord('Q'), 27):   # Q or Esc
            print("Quitting.")
            break
        elif key in (ord('s'), ord('S')):     # S → save snapshot
            ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
            path = SAVE_DIR / f"snapshot_{ts}.png"
            cv2.imwrite(str(path), frame)
            print(f"Saved snapshot → {path}")

finally:
    cap.release()
    cv2.destroyAllWindows()

print("Camera released.")

---
# 12. Closing Notes

## The five deliverables, mapped to code

| Deliverable | Where |
|-------------|-------|
| 1. Corn detection model | §4 trains `yolo11-pose`; §5 validates it |
| 2. Position + orientation per plant | §6 returns stem + leaf axis per plant |
| 3. Optimal growth line per row | §8: `OptimalLine.from_point_angle()` if known, `from_ransac()` otherwise |
| 4. Deviation from optimal | §9 computes positional (pixels) + angular (degrees) deviation |
| 5. Plant and row alignment scores | §9 scoring + §11 CSV export |

## Four "important aspects"

- **How is the optimal line defined?** For our greenhouse: hand-planted perpendicular rows → use `OptimalLine.from_point_angle()` with known coordinates. For general fields: RANSAC fit from detected stems.
- **Planting-accuracy consistency?** Read the per-plant deviations in `plants.csv` — the distribution gives you the answer empirically.
- **Detection accuracy needed for "helpful" outcomes?** §5.2 reports per-keypoint pixel error. Keep stem error below `POS_TOLERANCE_PX / 3` so scoring noise is small vs. real planting noise.
- **Weather robustness?** §4.1's augmentation simulates varied lighting, occlusion, perspective. Real field images will be the real test.

## Knobs to re-tune as the project evolves

| Knob | Cell | What it controls |
|------|------|------------------|
| `POS_TOLERANCE_PX`  | §9 | Pixel offset at which positional score drops to 37% |
| `ANG_TOLERANCE_DEG` | §9 | Angle at which angular score hits 0 |
| `eps_px` in `assign_rows` | §7 | Half the inter-row spacing in pixels |
| `EXPECTED_LEAF_OFFSET_DEG` | §9 | 90° for perpendicular trial, 0° for along-row trial |
| `residual_threshold` in `from_ransac` | §8 | Planting-accuracy tolerance for line fitting |

## What's deliberately left out

- **Pixel-to-centimeter conversion.** Requires camera calibration. Add a single `PX_PER_CM` constant once measured.
- **GPS row references.** If seeder GPS tracks become available, plug them into `fit_optimal_lines(known_lines=...)`.